# Adaptive Attention U-Net for Brain MRI Segmentation

This notebook implements an Adaptive Attention U-Net model that combines CNN features with selective transformer-style attention in the decoder for improved medical image segmentation. The model is specifically designed for brain MRI segmentation tasks.

In [ ]:
# Import necessary libraries
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import cv2
from tqdm import tqdm

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Implementation of Adaptive Self-Attention Module
def adaptive_attention_block(inputs, num_filters):
    # Multi-head self-attention
    def self_attention(x, heads=4):
        batch_size, height, width, channels = x.shape
        
        # Split channels into multiple heads
        head_dim = channels // heads
        x_reshaped = tf.reshape(x, (-1, height * width, heads, head_dim))
        x_transposed = tf.transpose(x_reshaped, [0, 2, 1, 3])
        
        # Compute query, key, value
        query = layers.Dense(head_dim)(x_transposed)
        key = layers.Dense(head_dim)(x_transposed)
        value = layers.Dense(head_dim)(x_transposed)
        
        # Scaled dot-product attention
        attention_weights = tf.matmul(query, key, transpose_b=True)
        attention_weights = attention_weights / tf.math.sqrt(tf.cast(head_dim, tf.float32))
        attention_weights = tf.nn.softmax(attention_weights, axis=-1)
        
        # Apply attention to values
        attended = tf.matmul(attention_weights, value)
        attended = tf.transpose(attended, [0, 2, 1, 3])
        attended = tf.reshape(attended, (-1, height, width, channels))
        
        return attended
    
    # Generate attention map
    attention_output = self_attention(inputs)
    
    # Adaptive gating mechanism
    gate = layers.Conv2D(1, 1, activation='sigmoid')(inputs)
    
    # Combine attention with input features adaptively
    attended_features = attention_output * gate
    output = layers.Add()([inputs, attended_features])
    
    return output

# Convolutional block
def conv_block(inputs, num_filters):
    x = layers.Conv2D(num_filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(num_filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    return x

In [ ]:
# Build the Adaptive Attention U-Net model
def build_adaptive_attention_unet(input_shape=(256, 256, 1)):
    # Input
    inputs = Input(input_shape)
    
    # Encoder
    # Contracting Path
    conv1 = conv_block(inputs, 64)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)
    
    conv2 = conv_block(pool1, 128)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)
    
    conv3 = conv_block(pool2, 256)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)
    
    conv4 = conv_block(pool3, 512)
    pool4 = layers.MaxPooling2D(pool_size=(2, 2))(conv4)
    
    # Bridge
    conv5 = conv_block(pool4, 1024)
    
    # Decoder
    # Expansive Path with Adaptive Attention
    up6 = layers.Conv2DTranspose(512, 2, strides=(2, 2), padding='same')(conv5)
    up6 = layers.concatenate([up6, conv4])
    up6 = adaptive_attention_block(up6, 512)
    conv6 = conv_block(up6, 512)
    
    up7 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(conv6)
    up7 = layers.concatenate([up7, conv3])
    up7 = adaptive_attention_block(up7, 256)
    conv7 = conv_block(up7, 256)
    
    up8 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(conv7)
    up8 = layers.concatenate([up8, conv2])
    up8 = adaptive_attention_block(up8, 128)
    conv8 = conv_block(up8, 128)
    
    up9 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv8)
    up9 = layers.concatenate([up9, conv1])
    up9 = adaptive_attention_block(up9, 64)
    conv9 = conv_block(up9, 64)
    
    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv9)
    
    model = Model(inputs=[inputs], outputs=[outputs])
    return model

# Define loss functions and metrics
def dice_coefficient(y_true, y_pred):
    smooth = 1e-15
    y_true = tf.keras.backend.flatten(y_true)
    y_pred = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true) + tf.keras.backend.sum(y_pred) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

# Combined loss function
def combined_loss(y_true, y_pred):
    return 0.5 * tf.keras.losses.binary_crossentropy(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

In [ ]:
# Create data generators for training
def create_data_generators(image_paths, mask_paths, batch_size=8):
    def load_and_preprocess(image_path, mask_path):
        # Load image
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (256, 256))
        image = image / 255.0
        image = np.expand_dims(image, axis=-1)
        
        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (256, 256))
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=-1)
        
        return image, mask
    
    def data_generator(image_paths, mask_paths, batch_size):
        num_samples = len(image_paths)
        while True:
            indices = np.random.permutation(num_samples)
            for i in range(0, num_samples, batch_size):
                batch_indices = indices[i:i + batch_size]
                batch_images = []
                batch_masks = []
                
                for idx in batch_indices:
                    image, mask = load_and_preprocess(image_paths[idx], mask_paths[idx])
                    batch_images.append(image)
                    batch_masks.append(mask)
                
                yield np.array(batch_images), np.array(batch_masks)
    
    return data_generator(image_paths, mask_paths, batch_size)

# Define evaluation metrics
def evaluate_segmentation(model, test_images, test_masks):
    predictions = model.predict(test_images)
    predictions = (predictions > 0.5).astype(np.uint8)
    
    dice_scores = []
    iou_scores = []
    
    for i in range(len(test_images)):
        # Calculate Dice coefficient
        intersection = np.sum(predictions[i] * test_masks[i])
        dice = (2. * intersection) / (np.sum(predictions[i]) + np.sum(test_masks[i]))
        dice_scores.append(dice)
        
        # Calculate IoU
        union = np.sum(predictions[i]) + np.sum(test_masks[i]) - intersection
        iou = intersection / union if union > 0 else 0
        iou_scores.append(iou)
    
    return {
        'mean_dice': np.mean(dice_scores),
        'std_dice': np.std(dice_scores),
        'mean_iou': np.mean(iou_scores),
        'std_iou': np.std(iou_scores)
    }

In [ ]:
# Model training and evaluation
def train_and_evaluate_model(train_images, train_masks, val_images, val_masks, batch_size=8, epochs=50):
    # Initialize model
    model = build_adaptive_attention_unet()
    
    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=combined_loss,
        metrics=[dice_coefficient, 'accuracy']
    )
    
    # Create data generators
    train_generator = create_data_generators(train_images, train_masks, batch_size)
    val_generator = create_data_generators(val_images, val_masks, batch_size)
    
    # Setup callbacks
    callbacks = [
        ModelCheckpoint(
            'best_model.h5',
            save_best_only=True,
            monitor='val_dice_coefficient',
            mode='max'
        ),
        EarlyStopping(
            monitor='val_dice_coefficient',
            patience=10,
            mode='max',
            restore_best_weights=True
        )
    ]
    
    # Train model
    history = model.fit(
        train_generator,
        steps_per_epoch=len(train_images) // batch_size,
        epochs=epochs,
        validation_data=val_generator,
        validation_steps=len(val_images) // batch_size,
        callbacks=callbacks
    )
    
    return model, history

# Visualization functions
def plot_training_history(history):
    plt.figure(figsize=(12, 4))
    
    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot Dice coefficient
    plt.subplot(1, 2, 2)
    plt.plot(history.history['dice_coefficient'], label='Training Dice')
    plt.plot(history.history['val_dice_coefficient'], label='Validation Dice')
    plt.title('Dice Coefficient')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Coefficient')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

def visualize_predictions(model, test_images, test_masks, num_samples=5):
    predictions = model.predict(test_images[:num_samples])
    predictions = (predictions > 0.5).astype(np.uint8)
    
    plt.figure(figsize=(15, 5))
    for i in range(num_samples):
        # Original image
        plt.subplot(3, num_samples, i + 1)
        plt.imshow(test_images[i, ..., 0], cmap='gray')
        plt.title('Original')
        plt.axis('off')
        
        # Ground truth
        plt.subplot(3, num_samples, i + 1 + num_samples)
        plt.imshow(test_masks[i, ..., 0], cmap='gray')
        plt.title('Ground Truth')
        plt.axis('off')
        
        # Prediction
        plt.subplot(3, num_samples, i + 1 + 2*num_samples)
        plt.imshow(predictions[i, ..., 0], cmap='gray')
        plt.title('Prediction')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Model Description and Features

The implemented Adaptive Attention U-Net combines the strengths of CNNs and Transformers with the following key features:

1. **Adaptive Self-Attention**: Selectively applies attention mechanisms in decoder layers to balance local and global feature learning.
2. **Gating Mechanism**: Uses learnable gates to control the influence of attention features.
3. **Multi-head Attention**: Implements parallel attention heads for capturing different types of relationships in the feature space.
4. **Skip Connections**: Preserves fine-grained spatial information through U-Net-style skip connections.

## Evaluation Metrics

The model uses several metrics to evaluate segmentation performance:

1. **Dice Coefficient**: Measures overlap between predicted and ground truth segmentations
2. **IoU (Intersection over Union)**: Evaluates segmentation accuracy
3. **Binary Cross-Entropy**: Measures pixel-wise classification accuracy
4. **Combined Loss**: Balances BCE and Dice loss for optimal training

## Training Features

- Learning rate scheduling
- Early stopping
- Model checkpointing
- Batch normalization
- Data augmentation capabilities

In [ ]:
# Example usage (with dummy data for demonstration)
# Replace this with your actual data loading code

# Generate dummy data for demonstration
def create_dummy_data(num_samples=100):
    images = np.random.rand(num_samples, 256, 256, 1)
    masks = (np.random.rand(num_samples, 256, 256, 1) > 0.5).astype(np.float32)
    return images, masks

# Create dummy dataset
X, y = create_dummy_data()

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Train model
model, history = train_and_evaluate_model(
    X_train, y_train,
    X_val, y_val,
    batch_size=8,
    epochs=50
)

# Plot training history
plot_training_history(history)

# Evaluate model
evaluation_metrics = evaluate_segmentation(model, X_test, y_test)
print("\nTest Set Evaluation:")
print(f"Mean Dice Coefficient: {evaluation_metrics['mean_dice']:.4f} ± {evaluation_metrics['std_dice']:.4f}")
print(f"Mean IoU: {evaluation_metrics['mean_iou']:.4f} ± {evaluation_metrics['std_iou']:.4f}")

# Visualize some predictions
visualize_predictions(model, X_test, y_test, num_samples=5)